# Adaptive spatial sampling lab
Select candidate field locations using predictive uncertainty and spatial diversity. Then compare a simple UCB1 policy for allocating repeated survey effort.

In [ ]:
from pathlib import Path
import sys
relative_package = Path('implementations/python/adaptive-sampling')
repo_root = next((parent for parent in (Path.cwd(), *Path.cwd().parents)
                  if (parent / relative_package).is_dir()), None)
if repo_root is None:
    raise RuntimeError('Could not locate implementations/python/adaptive-sampling from notebook working directory')
sys.path.insert(0, str(repo_root / relative_package))
from sampling import select_spatial_batch, ucb1_arm
candidate_xy = [(0,0), (0.1,0), (5,0), (10,0), (10.1,0)]
uncertainty = [0.95, 0.9, 0.7, 0.8, 0.75]
select_spatial_batch(candidate_xy, uncertainty, 3, diversity_weight=1.0)

## Field constraints
A minimum visit separation may make the requested batch infeasible. The selector returns a smaller batch rather than silently violating the constraint. Use projected metric coordinates for local distance calculations.

In [ ]:
select_spatial_batch(candidate_xy, uncertainty, 4, min_distance=2.0)

## UCB1 allocation
Counts and cumulative rewards summarize historical outcomes for each survey/annotation arm. Untried arms are explored first. Classical UCB1 assumes a stationary reward process; operational survey costs and changing conditions may require contextual or cost-aware alternatives.

## Finite-candidate Gaussian-process optimization
Fit a small RBF Gaussian process to observed locations and choose the next candidate using an acquisition function. Dense GP fitting is cubic in the number of observations; this is a teaching baseline, not a large-scale spatial optimizer.

In [ ]:
from sampling import gp_select_next
observed_xy = [(0.0,), (1.0,), (2.0,)]
observed_value = [0.1, 0.8, 0.3]
candidate_xy = [(0.5,), (1.5,), (2.5,), (3.0,)]
gp_select_next(observed_xy, observed_value, candidate_xy, acquisition='ei')

In [ ]:
counts = [12, 8, 0]
cumulative_rewards = [8.5, 6.0, 0.0]
ucb1_arm(counts, cumulative_rewards)

## Design safeguards
- Keep the sampling frame and selection probabilities.
- Separate model-driven active samples from probability samples used for unbiased accuracy estimation.
- Track travel cost, access constraints and spatial clustering.
- Refit uncertainty after new observations; do not treat initial scores as fixed.